In [13]:
#!unzip -o data.zip -d data
#! pip install open3d

In [ ]:
!nvidia-smi



Thu Jun 11 13:13:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 610.43.02              KMD Version: 610.43.02     CUDA UMD Version: 13.3     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4070 Ti     On  |   00000000:01:00.0  On |                  N/A |
|  0%   41C    P2             37W /  285W |    5014MiB /  12282MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("Device count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
Device count: 1
GPU: NVIDIA GeForce RTX 4070 Ti


In [35]:
import os

size_bytes = os.path.getsize("data.zip")
size_gb = size_bytes / (1024 ** 3)

print(f"Size: {size_gb:.2f} GB")

Size: 3.77 GB


In [2]:
!rm -rf data

In [2]:
from ultralytics import YOLO
import cv2
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import shutil   
import torch

In [6]:
# ── Config ─────────────────────────────────────────────────────────────────────
MODELS = ["yolov8n-seg.pt", "yolo11n-seg.pt", "yolo26n-seg.pt"]
TRAIN_CFG = dict(
    data="dataset/data.yaml",
    epochs=300,
    imgsz=640,
    batch=50,
    workers=10,
    device=0,  # RTX 4070 Ti
    project="runs/version_compare",
    exist_ok=True,
    degrees=10,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    mosaic=0.5,
    copy_paste=0.2,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4
)
OUTPUT_PATH = Path("runs/version_compare/comparison.csv")

# ── Train & collect ────────────────────────────────────────────────────────────
rows = []

for model_name in MODELS:
    tag = model_name.replace(".pt", "")
    print(f"\n{'='*60}\nTraining: {model_name}\n{'='*60}")
    try:
        results = YOLO(model_name).train(**TRAIN_CFG, name=tag)
        df = pd.read_csv(Path(results.save_dir) / "results.csv")
        df.columns = df.columns.str.strip()

        row = {"model": model_name, "status": "success", **df.iloc[-1].to_dict()}
        rows.append(row)
        print(f"{model_name} done")

    except Exception as e:
        rows.append({"model": model_name, "status": f"FAILED: {e}"})
        print(f"{model_name} failed: {e}")

# ── Save ───────────────────────────────────────────────────────────────────────
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
pd.DataFrame(rows).to_csv(OUTPUT_PATH, index=False, encoding="utf-8")
print(f"\n Saved → {OUTPUT_PATH.resolve()}")



Training: yolov8n-seg.pt
New https://pypi.org/project/ultralytics/8.4.65 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.56 🚀 Python-3.12.3 torch-2.12.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 11894MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=50, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.2, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset/data.yaml, degrees=10, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-seg.pt, momentum=0.937, mosa

Exception in thread Thread-96 (_pin_memory_loop):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1073, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1010, in run
    self._target(*self._args, **self._kwargs)
  File "/home/icp-group2/Documents/modeltrain/venv/lib/python3.12/site-packages/torch/utils/data/_utils/pin_memory.py", line 52, in _pin_memory_loop
    do_one_step()
  File "/home/icp-group2/Documents/modeltrain/venv/lib/python3.12/site-packages/torch/utils/data/_utils/pin_memory.py", line 28, in do_one_step
    r = in_queue.get(timeout=MP_STATUS_CHECK_INTERVAL)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/queues.py", line 122, in get
    return _ForkingPickler.loads(res)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/icp-group2/Documents/modeltrain/venv/lib/python3.12/site-packages/torch/multiprocessing/reductions.py", line 540, in rebuild_storage_fd


      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss   sem_loss  Instances       Size
      1/300      5.62G      2.199      5.027      10.47    0.01937      6.192         21        640: 100% ━━━━━━━━━━━━ 44/44 4.3it/s 10.3s0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 3.7it/s 0.8s0.6s
                   all        235        235    0.00526      0.797     0.0682     0.0188    0.00321      0.597     0.0534     0.0114

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss   sem_loss  Instances       Size
      2/300      5.14G      1.914      4.073      7.659    0.01558      3.286         21        640: 100% ━━━━━━━━━━━━ 44/44 6.1it/s 7.2s0.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 4.3it/s 0.7s0.6s
                   all        235 

In [5]:
model = YOLO("runs/segment/runs/version_compare/yolo26n-seg/weights/best.pt")

results = model.predict(
    source="dataset/test/images",  
    imgsz=640,
    save=True
)



image 1/235 /home/icp-group2/Documents/modeltrain/dataset/test/images/img0003.png: 640x480 1 copper, 46.7ms
image 2/235 /home/icp-group2/Documents/modeltrain/dataset/test/images/img0005.png: 640x480 1 copper, 3.5ms
image 3/235 /home/icp-group2/Documents/modeltrain/dataset/test/images/img0026.png: 640x480 1 copper, 3.7ms
image 4/235 /home/icp-group2/Documents/modeltrain/dataset/test/images/img0033.png: 640x480 1 copper, 3.7ms
image 5/235 /home/icp-group2/Documents/modeltrain/dataset/test/images/img0040.png: 640x480 1 copper, 3.5ms
image 6/235 /home/icp-group2/Documents/modeltrain/dataset/test/images/img0041.png: 640x480 1 copper, 3.6ms
image 7/235 /home/icp-group2/Documents/modeltrain/dataset/test/images/img0048.png: 640x480 1 copper, 3.4ms
image 8/235 /home/icp-group2/Documents/modeltrain/dataset/test/images/img0053.png: 640x480 1 copper, 3.6ms
image 9/235 /home/icp-group2/Documents/modeltrain/dataset/test/images/img0062.png: 640x480 1 copper, 3.6ms
image 10/235 /home/icp-group2/Docum

In [ ]:
img = Image.open("runs/segment/predict/phenoBench_00001.jpg")
plt.imshow(img)
plt.axis("off")

In [3]:
import os
import zipfile

def compress_folder(folder_path, output_zip):
    """
    Compress a folder into a ZIP file.

    Args:
        folder_path (str): Path to the folder to compress
        output_zip (str): Name of the output ZIP file
    """
    with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(folder_path):
            for file in files:
                file_path = os.path.join(root, file)

                # Preserve folder structure inside ZIP
                arcname = os.path.relpath(file_path, folder_path)

                zipf.write(file_path, arcname)

    print(f"Folder '{folder_path}' compressed into '{output_zip}'")


# Example usage
folder_to_compress = "runs"
output_zip_file = "runs.zip"

compress_folder(folder_to_compress, output_zip_file)

Folder 'runs' compressed into 'runs.zip'


## RGB-FUSION

In [ ]:
# arr = np.load("/home/icp-group2/Documents/modeltrain/dataset/train/depth/img0001.png")
import open3d as o3d
import numpy as np

pcd = o3d.io.read_point_cloud(
    "/home/icp-group2/Documents/modeltrain/dataset/train/pointcloud/img0001.ply"
)

points = np.asarray(pcd.points)

print(points.shape)
print(points.dtype)
print(points[:5])

#########################################################################################################

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
(259242, 3)
float64
[[   -0.15273    -0.23387       0.479]
 [   -0.15304    -0.23534       0.482]
 [    -0.1524    -0.23534       0.482]
 [   -0.15207    -0.23583       0.483]
 [   -0.15143    -0.23583       0.483]]


In [15]:
import cv2
import numpy as np

depth = cv2.imread(
    "/home/icp-group2/Documents/modeltrain/dataset/train/depth/img0001.png",
    cv2.IMREAD_UNCHANGED
)

print(depth.shape)
print(depth.dtype)
print(depth.min(), depth.max())

(576, 640, 3)
uint8
0 255


In [ ]:
# ── Config ─────────────────────────────────────────────────────────────────────
DATASET_ROOT = Path("/home/icp-group2/Documents/modeltrain/dataset")
FUSED_ROOT   = Path("/home/icp-group2/Documents/modeltrain/dataset_rgbd")
MODELS       = ["yolov8n-seg.pt", "yolo11n-seg.pt", "yolo26n-seg.pt"]
OUTPUT_PATH  = Path("runs/version_compare_rgbd/comparison.csv")
IMG_SIZE     = 640
# ── Fuse RGB + Depth for train and val only ───────────────────────────
def fuse_dataset():
    for split in ["train", "val"]:
        rgb_dir   = DATASET_ROOT / split / "images"
        depth_dir = DATASET_ROOT / split / "depth"
        label_dir = DATASET_ROOT / split / "labels"

        out_img_dir   = FUSED_ROOT / split / "images"
        out_label_dir = FUSED_ROOT / split / "labels"
        out_img_dir.mkdir(parents=True, exist_ok=True)
        out_label_dir.mkdir(parents=True, exist_ok=True)

        skipped, fused = 0, 0

        for rgb_path in sorted([*rgb_dir.glob("*.png"), *rgb_dir.glob("*.jpg")]):
            stem = rgb_path.stem

            # Load depth — npy preferred over png
            depth_npy = depth_dir / f"{stem}.npy"
            depth_png = depth_dir / f"{stem}.png"

            if depth_npy.exists():
                depth_raw = np.load(str(depth_npy)).astype(np.float32)
            elif depth_png.exists():
                depth_raw = cv2.imread(str(depth_png), cv2.IMREAD_ANYDEPTH).astype(np.float32)
            else:
                print(f"No depth for {stem}, skipping")
                skipped += 1
                continue

            # Resize RGB
            rgb = cv2.resize(cv2.imread(str(rgb_path)), (IMG_SIZE, IMG_SIZE))

            # Normalize depth to 0-255
            d_min, d_max = np.nanmin(depth_raw), np.nanmax(depth_raw)
            depth_uint8  = ((depth_raw - d_min) / (d_max - d_min + 1e-8) * 255).astype(np.uint8)
            depth_resized = cv2.resize(depth_uint8, (IMG_SIZE, IMG_SIZE))

            # Save 4-channel fused image
            np.save(str(out_img_dir / f"{stem}.npy"), np.dstack([rgb, depth_resized]))

            # Copy label unchanged
            label_src = label_dir / f"{stem}.txt"
            if label_src.exists():
                shutil.copy(label_src, out_label_dir / f"{stem}.txt")

            fused += 1

        print(f"{split}: {fused} fused, {skipped} skipped")

print("Fusing RGB + Depth...")
fuse_dataset()

NameError: name 'Path' is not defined

## YOLO-SAM-MODEL MULITPLE-OBJ-LABELLING

In [16]:
from ultralytics import YOLO

model = YOLO("/home/icp-group2/Documents/modeltrain/runs/segment/runs/version_compare/yolov8n-seg/weights/best.pt")


In [4]:
from ultralytics import YOLO

model = YOLO("/home/icp-group2/Documents/modeltrain/runs/segment/runs/version_compare/yolov8n-seg/weights/best.pt")
data_path = "dataset-mul-images/images"

results = model.predict(
    source=data_path,
    save_txt=True,
    conf=0.05,
    iou=0.70,
    save=True
)


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

image 1/1811 /home/icp-group2/Documents/modeltrain/dataset-mul-images/images/img0000.png: 608x640 2 steels, 2.8ms
image 2/1811 /home/icp-group2/Documents/modeltrain/dataset-mul-images/images/img0001.png: 608x640 1 copper, 2.7ms
image 3/1811 /home/icp-group2/Documents/modeltrain/dataset-mul-images/images/img0002.png: 608x640 3 coppers, 2.3ms
image 4/1811 /home/icp-group2/Documents/modeltrain/dataset-mul-images/images/img0003.png: 608x640 3 coppers, 2.3ms


In [20]:
from ultralytics.data.annotator import auto_annotate

auto_annotate(
    data="dataset-mul-images/images",
    det_model=model,
    sam_model="sam_b.pt"
)


image 1/1811 /home/icp-group2/Documents/modeltrain/dataset-mul-images/images/img0000.png: 608x640 1 steel, 2.6ms
image 2/1811 /home/icp-group2/Documents/modeltrain/dataset-mul-images/images/img0001.png: 608x640 1 copper, 2.4ms
image 3/1811 /home/icp-group2/Documents/modeltrain/dataset-mul-images/images/img0002.png: 608x640 (no detections), 2.5ms
image 4/1811 /home/icp-group2/Documents/modeltrain/dataset-mul-images/images/img0003.png: 608x640 1 copper, 2.7ms
image 5/1811 /home/icp-group2/Documents/modeltrain/dataset-mul-images/images/img0004.png: 608x640 1 copper, 2.3ms
image 6/1811 /home/icp-group2/Documents/modeltrain/dataset-mul-images/images/img0005.png: 608x640 1 copper, 2.5ms
image 7/1811 /home/icp-group2/Documents/modeltrain/dataset-mul-images/images/img0006.png: 608x640 1 copper, 2.4ms
image 8/1811 /home/icp-group2/Documents/modeltrain/dataset-mul-images/images/img0007.png: 608x640 2 coppers, 2.4ms
image 9/1811 /home/icp-group2/Documents/modeltrain/dataset-mul-images/images/img

In [21]:
from ultralytics import YOLO

model = YOLO(model)

results = model("dataset-mul-images/images", save=True)


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

image 1/1811 /home/icp-group2/Documents/modeltrain/dataset-mul-images/images/img0000.png: 608x640 1 steel, 2.8ms
image 2/1811 /home/icp-group2/Documents/modeltrain/dataset-mul-images/images/img0001.png: 608x640 1 copper, 2.5ms
image 3/1811 /home/icp-group2/Documents/modeltrain/dataset-mul-images/images/img0002.png: 608x640 (no detections), 2.4ms
image 4/1811 /home/icp-group2/Documents/modeltrain/dataset-mul-images/images/img0003.png: 608x640 1 copper, 2.

In [22]:
results = model("/home/icp-group2/Documents/modeltrain/dataset-mul-images/images/img0000.png")
print(len(results[0].boxes))


image 1/1 /home/icp-group2/Documents/modeltrain/dataset-mul-images/images/img0000.png: 608x640 1 steel, 3.0ms
Speed: 11.2ms preprocess, 3.0ms inference, 0.8ms postprocess per image at shape (1, 3, 608, 640)
1


In [16]:
from ultralytics.models.sam import SAM3SemanticPredictor

# Initialize predictor with configuration
overrides = dict(
    conf=0.25,
    task="segment",
    mode="predict",
    model="sam3.pt",
    half=True,  # Use FP16 for faster inference
    save=True,
)
predictor = SAM3SemanticPredictor(overrides=overrides)

# Set image once for multiple queries
predictor.set_image("/home/icp-group2/Documents/modeltrain/dataset-mul-images/images/img0852.png")

# Query with multiple text prompts
results = predictor(text=["object on conveyor belt"])

# Works with descriptive phrases
results = predictor(text=["objects on conveyor belt", "3D objects on conveyor belt"])

# Query with a single concept
# results = predictor(text=["a person"])

Ultralytics 8.4.56 🚀 Python-3.12.3 torch-2.12.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 11903MiB)
WARNING ⚠️ imgsz=[640] must be multiple of max stride 14, updating to [644]

image 1/1 /home/icp-group2/Documents/modeltrain/dataset-mul-images/images/img0852.png: 644x644 4 object on conveyor belts, 19.8ms
Speed: 1.2ms preprocess, 19.8ms inference, 0.5ms postprocess per image at shape (1, 3, 644, 644)
Results saved to /home/icp-group2/Documents/modeltrain/runs/segment/predict-13

image 1/1 /home/icp-group2/Documents/modeltrain/dataset-mul-images/images/img0852.png: 644x644 4 objects on conveyor belts, 4 3D objects on conveyor belts, 29.9ms
Speed: 1.2ms preprocess, 29.9ms inference, 0.5ms postprocess per image at shape (1, 3, 644, 644)
Results saved to /home/icp-group2/Documents/modeltrain/runs/segment/predict-13


In [19]:
# results[0].masks
results[0].boxes

ultralytics.engine.results.Boxes object with attributes:

cls: tensor([0., 0., 0., 0., 1., 1., 1., 1.], device='cuda:0', dtype=torch.float16)
conf: tensor([0.9844, 0.9834, 0.9771, 0.9746, 0.9141, 0.9106, 0.8926, 0.8779], device='cuda:0', dtype=torch.float16)
data: tensor([[8.2100e+02, 4.0350e+02, 1.0040e+03, 6.6450e+02, 9.8438e-01, 0.0000e+00],
        [3.8700e+02, 6.7438e+01, 7.0000e+02, 3.2850e+02, 9.8340e-01, 0.0000e+00],
        [1.0450e+02, 3.6100e+02, 2.9150e+02, 5.4950e+02, 9.7705e-01, 0.0000e+00],
        [3.2800e+02, 3.2125e+02, 5.9850e+02, 6.4850e+02, 9.7461e-01, 0.0000e+00],
        [8.2050e+02, 4.0300e+02, 1.0045e+03, 6.6500e+02, 9.1406e-01, 1.0000e+00],
        [3.8750e+02, 6.7312e+01, 6.9950e+02, 3.2925e+02, 9.1064e-01, 1.0000e+00],
        [1.0375e+02, 3.6125e+02, 2.9150e+02, 5.4900e+02, 8.9258e-01, 1.0000e+00],
        [3.2725e+02, 3.2050e+02, 5.9800e+02, 6.4850e+02, 8.7793e-01, 1.0000e+00]], device='cuda:0', dtype=torch.float16)
id: None
is_track: False
orig_shape: (10

In [25]:
from ultralytics.models.sam import SAM3SemanticPredictor
from ultralytics import YOLO
import cv2
import numpy as np

sam = SAM3SemanticPredictor(overrides={
    "model": "sam3.pt",
    "task": "segment"
})

clf = YOLO("/home/icp-group2/Documents/modeltrain/runs/segment/runs/version_compare/yolo26n-seg/weights/best.pt")

img_path = "/home/icp-group2/Documents/modeltrain/dataset-mul-images/images/img0027.png"
img = cv2.imread(img_path)

# STEP 1: SAM segmentation
sam.set_image(img_path)
results = sam(text=["object on conveyor belt"])

masks = results[0].masks.data.cpu().numpy()

print(f"SAM found {len(masks)} objects")

# STEP 2: loop through SAM objects
for i, mask in enumerate(masks):

    mask = mask.astype(np.uint8)

    ys, xs = np.where(mask > 0)
    if len(xs) == 0:
        continue

    x1, x2 = xs.min(), xs.max()
    y1, y2 = ys.min(), ys.max()

    crop = img[y1:y2, x1:x2]

    # STEP 3: classification (THIS is missing in your current setup)
    pred = clf(crop)

    if pred[0].boxes is None or len(pred[0].boxes) == 0:
        continue

    cls_id = int(pred[0].boxes.cls[0])
    label = pred[0].names[cls_id]

    print(f"Object {i}: {label}")

    # STEP 4: draw result
    cv2.rectangle(img, (x1,y1), (x2,y2), (0,255,0), 2)
    cv2.putText(img, label, (x1,y1-10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,0), 2)

cv2.imwrite("final_output.jpg", img)

Ultralytics 8.4.56 🚀 Python-3.12.3 torch-2.12.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 11903MiB)
WARNING ⚠️ imgsz=[640] must be multiple of max stride 14, updating to [644]

image 1/1 /home/icp-group2/Documents/modeltrain/dataset-mul-images/images/img0027.png: 644x644 4 object on conveyor belts, 32.6ms
Speed: 1.2ms preprocess, 32.6ms inference, 0.5ms postprocess per image at shape (1, 3, 644, 644)
Results saved to /home/icp-group2/Documents/modeltrain/runs/segment/predict-18
SAM found 4 objects

0: 640x480 (no detections), 3.7ms
Speed: 0.8ms preprocess, 3.7ms inference, 0.1ms postprocess per image at shape (1, 3, 640, 480)

0: 640x384 (no detections), 48.0ms
Speed: 0.8ms preprocess, 48.0ms inference, 0.1ms postprocess per image at shape (1, 3, 640, 384)

0: 640x288 (no detections), 47.4ms
Speed: 0.5ms preprocess, 47.4ms inference, 0.1ms postprocess per image at shape (1, 3, 640, 288)

0: 512x640 (no detections), 47.2ms
Speed: 0.7ms preprocess, 47.2ms inference, 0.1ms postprocess per

True

In [ ]:
from ultralytics.models.sam import SAM3SemanticPredictor

# Initialize predictor with configuration
overrides = dict(
    conf=0.25,
    task="segment",
    mode="predict",
    model="sam3.pt",
    half=True,  # Use FP16 for faster inference
    save=True,
)
predictor = SAM3SemanticPredictor(overrides=overrides)

# Set image once for multiple queries
predictor.set_image("path/to/image.jpg")

# Query with multiple text prompts
results = predictor(text=["person", "bus", "glasses"])

# Works with descriptive phrases
results = predictor(text=["person with red cloth", "person with blue cloth"])

# Query with a single concept
results = predictor(text=["a person"])